# Chapter 12 — Retrieval Is a Policy

**Book alignment:** Embeddings From First Principles, Chapter 12

**Question this notebook isolates:** "We use model X for retrieval" describes one stage of
eight (query transform → representation → candidates → similarity → threshold → rerank →
diversity → context assembly). Does layering a policy on a *fixed* model move quality — and
on RELATE's near-restatement queries, does it help or hurt (the committed Wave 1 ablation)?

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave, name):
    sub = "wave1/artifacts/v02" if wave == "wave1-v02" else f"{wave}/artifacts"
    return json.loads((EXP / sub / name).read_text())

## 1. The policy is the artifact — one model, three policies (Wave 1)

In [ ]:
pa = art("wave1", "policy-ablation.json")["policies"]
for name in ("dense", "hybrid", "rerank"):
    v = pa[name]
    print(f"  {name:8} nDCG@10 all={v['ndcg10_all']:.4f}   hard-neg={v['ndcg10_hardneg']:.4f}")

# on RELATE v0.1, layering HURT: BM25 injected lexical noise; a generic NLI cross-encoder hurt more
assert pa["hybrid"]["ndcg10_all"] < pa["dense"]["ndcg10_all"]
assert pa["rerank"]["ndcg10_all"] < pa["hybrid"]["ndcg10_all"]
print("\nRELATE's queries are near-restatements -> dense already saturates -> no headroom for the policy layer")
print("the LITERATURE shows hybrid + rerank gains on harder sets; the structural point holds regardless:")
print("'we use model X' names 1 stage of 8")

## 2. Chunking is a retrieval parameter, not preprocessing

In [ ]:
# a fact that spans two chunks retrieves from neither; the blend weight is a tuned knob
DOC = "the refund policy allows returns within thirty days " * 3 + "unless the item was final sale"
def chunks(text, size):
    words = text.split()
    return [" ".join(words[i:i + size]) for i in range(0, len(words), size)]

small = chunks(DOC, 4)      # "final sale" clause split from "refund policy"
large = chunks(DOC, 40)
query_terms = {"refund", "final", "sale"}
best_small = max(small, key=lambda c: len(query_terms & set(c.split())))
best_large = max(large, key=lambda c: len(query_terms & set(c.split())))
print("best small chunk:", repr(best_small))
print("best large chunk covers all query terms:", query_terms <= set(best_large.split()))
assert not (query_terms <= set(best_small.split())) and (query_terms <= set(best_large.split()))
print("chunk boundaries decide whether the answer is retrievable at all - ablate them")

## What we earned

Retrieval is an eight-stage policy from query transform to context assembly; each stage has
parameters and a failure mode. The policy — every parameter bound — is what you version,
test, and roll back, not the embedding model. On RELATE v0.1, policy layering *hurt*
(no hard-case headroom); the structural point stands: two teams "using the same model" can
have entirely different retrieval behaviour.

**Notebook 13 / Chapter 13** asks how to evaluate the representation itself — and how far a
generic leaderboard number transfers to your problem.